# Этап 4: Grad-CAM (автономный)

Setup + auto-fetch dataset + при отсутствии модели сначала тренирует IrResnet4.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Lamblador/IR_expert_system_3.git"  # при необходимости замените
REPO_DIR = Path("IR_expert_system_3")

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd IR_expert_system_3
!pip install -q -e ".[torch]"
!ir-pipeline run list


In [ ]:
from pathlib import Path

DATASET_DIR = Path('data/processed/dataset_mini')
if not DATASET_DIR.exists():
    print('dataset_mini not found → fetching from HF...')
    !ir-pipeline fetch-data --filename dataset_mini.zip --extract-to data/processed
else:
    print(f'{DATASET_DIR} already exists')


In [ ]:
from pathlib import Path
from IPython.display import Image, display

PIPELINE_RUN = Path('runs/colab_pipeline_cam')
bundles = sorted(Path('runs').rglob('irresnet_bundle.pt'))
if not bundles:
    print('No irresnet bundle found, training one...')
    !ir-pipeline run stage train_irresnet --paths configs/paths.huggingface.yaml --pipeline-run {PIPELINE_RUN}
    bundles = sorted(Path('runs').rglob('irresnet_bundle.pt'))

IR_RUN = bundles[-1].parent
print('Using', IR_RUN)
!ir-pipeline cam-examples --paths configs/paths.huggingface.yaml --run-dir {IR_RUN} --output-dir reports/colab_cam

for p in sorted(Path('reports/colab_cam').glob('cam_example_*.png'))[:3]:
    display(Image(filename=str(p), width=900))
